<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Multi-Agent Orchestration with The OpenAI Agents SDK on Databricks
#### Duration: 35 minutes
## Overview
In this lab, you will apply what you learned in the *Introduction to OpenAI Agents SDK on Databricks* demo to build a multi-agent system yourself. You will configure the SDK, connect two worker agents to Unity Catalog functions through Managed MCP servers, create a supervisor that routes requests with handoffs, refine the supervisor's routing instructions in its YAML config, and trace the full orchestration with MLflow.

Each step has a **TODO** with `<FILL_IN>` placeholders. The instructions above each TODO tell you what to fill in, and you can reveal the answer if you get stuck.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Configure the OpenAI Agents SDK** to route requests through Databricks model services
2. **Connect worker agents to Unity Catalog functions** using Managed MCP servers, with `DatabricksMCPClient` for discovery and `McpServer` for agent integration
3. **Build a supervisor agent** that delegates user requests to worker agents with `handoffs`
4. **Validate and refine routing** by testing queries and editing the supervisor's YAML instructions
5. **Capture multi-agent execution** with MLflow autologging
6. **Combine spans into a single trace** with `@mlflow.trace` and `mlflow.start_span`

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-6

Installing 11 package(s) with uv...
  pyyaml==6.0.3
  databricks-openai==0.17.1
  databricks-agents==1.12.0
  databricks-sdk==0.145.0
  openai==3.22.1
  openai-agents==0.22.3
  nest_asyncio==1.6.0
  databricks-mcp==0.9.2
  mlflow==3.16.1
  mlflow-skinny==3.16.1
  mlflow-tracing==3.16.1
Library installation complete.


Starting Demo Environment Setup
Using config: Includes/config/config-6.yaml

[0] Compute check passed: Serverless 6 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17579966_1791918748' found. Using it.

[2] Setting up schema...
  Catalog : labuser17579966_1791918748
  Schema  : agent_training

[3] Setting up Unity Gateway model service...
  Model service already exists and matches: labuser17579966_1791918748.agent_training.agent-training-model → databricks-gpt-5-2

[4] Processing datasets...
  Using Vocareum Delta Share: airbnb

[5] Registering 1 SQL UC tool(s)...

Creating Tools
  UC function created: neighborhood_summary (from sql/)

[6] Registering 1 Python UC tool(s)...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-159aa9ec-c48d-4fb6-9088-bd39fbd588dc/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


  Registered Python UC function: classify_price_tier

[7] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: oai-ma-sdk-training
  Created experiment: /Workspace/Users/labuser17579966_1791918748@vocareum.com/oai-ma-sdk-training
  Active experiment: /Workspace/Users/labuser17579966_1791918748@vocareum.com/oai-ma-sdk-training
Refreshing environment

Demo Environment Setup Complete


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Catalog,"labuser17579966_1791918748<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Schema,agent_trainingCopy
Table,"labuser17579966_1791918748.agent_training.sf_airbnb_listings<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.sf_airbnb_listings';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
SQL function,"labuser17579966_1791918748.agent_training.neighborhood_summary<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.neighborhood_summary';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Python function,"labuser17579966_1791918748.agent_training.classify_price_tier<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.classify_price_tier';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Model service,"labuser17579966_1791918748.agent_training.agent-training-model<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.agent-training-model';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
MLflow experiment,"/Workspace/Users/labuser17579966_1791918748@vocareum.com/oai-ma-sdk-training<button onclick=""var t=document.createElement('textarea');t.value='/Workspace/Users/labuser17579966_1791918748@vocareum.com/oai-ma-sdk-training';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"


### A1. Lab Scenario

<div class="scnB"><style>.scnB{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;color:#0b2026;width:100%;max-width:100%;box-sizing:border-box;margin:0;padding:8px}.scnB .scen{background:#FFFFFF;color:#0b2026;padding:26px 30px;border-radius:12px;box-sizing:border-box;overflow:hidden;box-shadow:0 2px 16px rgba(27,49,57,0.10),0 1px 4px rgba(27,49,57,0.06);border:1px solid rgba(27,49,57,0.08)}.scnB .kicker{font-size:14px;text-transform:uppercase;letter-spacing:1.2px;color:#5A6F77;margin-bottom:10px;font-weight:700;display:flex;align-items:center;gap:10px}.scnB .body{font-size:15px;line-height:1.7;color:#0b2026}.scnB .body p{margin:0 0 14px}.scnB .body p:last-child{margin:0}.scnB .tabs{display:flex;gap:6px;margin-bottom:0}.scnB .tbtn{padding:9px 18px;border-radius:8px 8px 0 0;border:1px solid #DCE0E2;border-bottom:none;cursor:pointer;font-size:14px;font-weight:700;background:#EEEDE9;color:#5A6F77;font-family:inherit}.scnB .tbtn:hover{background:#DCE0E2;color:#1B3139}.scnB .tbtn-on{background:#2272B4;color:#FFFFFF;border-color:#2272B4}.scnB .tbtn-on:hover{background:#2272B4;color:#FFFFFF}.scnB .tstack{position:relative;display:grid}.scnB .tpane{grid-area:1 / 1 / 2 / 2;background:rgba(66,153,224,0.12);border:1px solid rgba(34,114,180,0.25);border-top:3px solid #2272B4;border-radius:0 8px 8px 8px;padding:18px 22px;font-size:15px;line-height:1.7;color:#0b2026;box-sizing:border-box;visibility:hidden;opacity:0;pointer-events:none}.scnB .tpane-on{visibility:visible;opacity:1;pointer-events:auto}.scnB code{background:rgba(27,49,57,0.07);padding:2px 6px;border-radius:4px;font-size:14px;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;color:#0b2026}.scnB ul{margin:8px 0 0 0;padding-left:20px}.scnB li{margin-bottom:10px}.scnB .eq{background:rgba(0,169,114,.10);border-left:4px solid #00A972;padding:10px 14px;border-radius:0 6px 6px 0;margin:10px 0 0;font-size:14px;font-style:italic;line-height:1.6;color:#0b2026}.scnB .eq-q{background:#FFF6F4;border-left:4px solid #FF5F46;padding:10px 14px;border-radius:0 6px 6px 0;margin:10px 0 0;font-size:14px;font-style:italic;line-height:1.6;color:#0b2026}</style><div class="scen">
<div class="kicker"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#5A6F77" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 8V4H8"/><rect x="4" y="8" width="16" height="12" rx="2"/><path d="M2 14h2"/><path d="M20 14h2"/><path d="M9 13v2"/><path d="M15 13v2"/></svg>Scenario</div>
<div class="body">
<p>We have been tasked with building a multi-agent POC for the data team. The supervisor agent routes requests to two specialized sub-agents: a data analyst and a business analyst. Each agent is equipped with a UC function accessed through MCP. Select a tab below to explore each agent's role.</p>
</div>
<div class="tabs">
<button onclick="showTabB('supervisor')" id="btnB-supervisor" class="tbtn tbtn-on">Supervisor</button>
<button onclick="showTabB('analyst')" id="btnB-analyst" class="tbtn">Data Analyst</button>
<button onclick="showTabB('business')" id="btnB-business" class="tbtn">Business Analyst</button>
</div>
<div class="tstack">
<div id="tpB-supervisor" class="tpane tpane-on">
<strong>Role:</strong> The supervisor agent is the entry point for all incoming requests. It does not hold any tools itself &#x2014; instead, it reads the intent of each request and routes it to the appropriate sub-agent.<br><br>
<strong>Routing logic:</strong>
<ul>
<li>Questions about <em>pricing tiers, deal evaluation, or price classification</em> &#x2192; routed to the <strong>Data Analyst</strong></li>
<li>Questions about <em>neighborhood summaries, average prices, listing counts, or room type mix</em> &#x2192; routed to the <strong>Business Analyst</strong></li>
</ul>
</div>
<div id="tpB-analyst" class="tpane">
<strong>Role:</strong> The data analyst agent classifies Airbnb listing prices into market tiers based on SF pricing distribution. It helps users understand whether a price point is Budget, Mid-Range, Premium, or Luxury.<br><br>
<strong>Tool:</strong> <code>classify_price_tier</code> (via MCP)<br><br>
<strong>Returns:</strong> A JSON object containing the price, tier, percentile band, and a short interpretation.<br><br>
<div class="eq-q">"Is $200/night a good deal for an Airbnb in SF?"</div>
</div>
<div id="tpB-business" class="tpane">
<strong>Role:</strong> The business analyst agent surfaces market insights for stakeholders. It aggregates data to answer strategic questions about neighborhoods, pricing, and listing mix.<br><br>
<strong>Tool:</strong> <code>neighborhood_summary</code> (via MCP)<br><br>
<strong>Returns:</strong> A table of per-neighborhood metrics including average price, total listings, and most common room type.<br><br>
<div class="eq-q">"Which neighborhoods have the highest average price?"</div>
</div>
</div>
</div></div>
<script>
function showTabB(name) {
  ['supervisor','analyst','business'].forEach(function(t) {
    var pane = document.getElementById('tpB-' + t);
    var btn = document.getElementById('btnB-' + t);
    if (t === name) {
      pane.className = 'tpane tpane-on';
      btn.className = 'tbtn tbtn-on';
    } else {
      pane.className = 'tpane';
      btn.className = 'tbtn';
    }
  });
}
</script>

### A2. Import Libraries and Configure the SDK

**TODO:** Configure the OpenAI Agents SDK the same way you did in notebook 05. Fill in the three `<FILL_IN>` placeholders to:
1. Set the default client to `AsyncDatabricksOpenAI`, routed through **Unity Gateway model services** (`use_ai_gateway=True`).
1. Set the default API to `"chat_completions"`.
1. Disable the OpenAI trace processor by passing an empty list, so MLflow handles tracing instead.

The imports, `WorkspaceClient`, and `nest_asyncio` setup are provided. As part of the classroom setup, a UC MLflow experiment has been created for you:
- UC table prefixes: `oai_ma_sdk`
- MLflow experiment: `oai-ma-sdk-training`

Additional documentation and reading:
- [`set_default_openai_client`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_client)
- [`set_default_openai_api`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_api)
- [`set_trace_processors`](https://github.com/openai/openai-agents-python/blob/main/docs/tracing.md)

In [0]:
from agents import Agent, Runner, handoff, set_default_openai_api, set_default_openai_client
from agents.tracing import set_trace_processors
from databricks_openai import AsyncDatabricksOpenAI
from databricks_openai.agents import McpServer
from databricks.sdk import WorkspaceClient

import asyncio
import nest_asyncio
import logging

nest_asyncio.apply()

## Route the SDK through Databricks model services (recall notebook 05, section A1)
# set_default_openai_client(<FILL_IN>)
# set_default_openai_api(<FILL_IN>)
# set_trace_processors(<FILL_IN>)  # Disable the OpenAI trace processor; MLflow handles tracing
set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))
set_default_openai_api("chat_completions")
set_trace_processors([])  # Disable the OpenAI trace processor; MLflow handles tracing
logging.getLogger("openai").setLevel(logging.WARNING)

ws = WorkspaceClient()
workspace_host = ws.config.host

print("OpenAI Agents SDK configured to use Databricks model routing.")
print(f"Workspace host: {workspace_host}")

OpenAI Agents SDK configured to use Databricks model routing.
Workspace host: https://dbc-358760f8-bd5d.cloud.databricks.com



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from agents import Agent, Runner, handoff, set_default_openai_api, set_default_openai_client
from agents.tracing import set_trace_processors
from databricks_openai import AsyncDatabricksOpenAI
from databricks_openai.agents import McpServer
from databricks.sdk import WorkspaceClient

import asyncio
import nest_asyncio
import logging

nest_asyncio.apply()

set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))
set_default_openai_api("chat_completions")
set_trace_processors([])  # Disable the OpenAI trace processor; MLflow handles tracing
logging.getLogger("openai").setLevel(logging.WARNING)

ws = WorkspaceClient()
workspace_host = ws.config.host

print("OpenAI Agents SDK configured to use Databricks model routing.")
print(f"Workspace host: {workspace_host}")
```

</div>
</details>

## B. Connect to UC Tools via MCP

### B1. Discover Available UC Functions

Before building agents, verify that the UC functions are registered and discoverable. This lab uses two ways to work with the Databricks-managed MCP server:
1. `DatabricksMCPClient`: a synchronous API for direct MCP tool calls (good for notebooks)
1. `McpServer`: an asynchronous context manager that connects MCP servers to OpenAI Agents SDK agents (useful for applications)

**TODO:** Build the MCP server URL using the pattern above, `{workspace_host}/api/2.0/mcp/functions/{catalog}/{schema}`, then list the tools it exposes. You should see both `classify_price_tier` and `neighborhood_summary`.

In [0]:
from databricks_mcp import DatabricksMCPClient

## Point to the Managed MCP server for the UC functions in your catalog and schema
# mcp_url = <FILL_IN>
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

## Discover the tools the MCP server exposes
# tools = mcp_client.<FILL_IN>
tools = mcp_client.list_tools()
print(f"MCP server URL: {mcp_url}")
print(f"Discovered {len(tools)} UC function tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description}")

MCP server URL: https://dbc-358760f8-bd5d.cloud.databricks.com/api/2.0/mcp/functions/labuser17579966_1791918748/agent_training
Discovered 3 UC function tool(s):
  - labuser17579966_1791918748__agent_training__classify_price_tier: Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for questions like: "Is $180/night a good deal in SF?", "What tier does this listing fall into?" Returns a JSON object with tier, percentile_band, and a short interpretation.
  - labuser17579966_1791918748__agent_training__classify_price_uc: Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for 


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks_mcp import DatabricksMCPClient

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()
print(f"MCP server URL: {mcp_url}")
print(f"Discovered {len(tools)} UC function tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description}")
```

</div>
</details>

### B2. Build MCP-Backed Agents

Each agent will have its own configuration file. Additionally, each agent will have its own `McpServer` scoped to a single UC function. This ensures each agent can only access the tool it needs:
1. `classify_price_tier` → **Data Analyst agent**
1. `neighborhood_summary` → **Business Analyst agent**

#### B2.1 Load the Agent Configuration
We will load the agent configuration file. This is a recommended practice since any updates to the agents' system prompts, name, etc. live outside this notebook. These variables will be stored in the dictionary `agents`:
- `agents["X"]['name']`: The agent's name
- `agents["X"]['llm_endpoint']`: The name of the Unity Catalog **model service** the agent calls (`agent-training-model`). In the next step you combine it with your catalog and schema into the service's fully qualified name.
- `agents["X"]['system_prompt']`: The agent's system prompt

Running the following cell will store config variables for the data analyst, business data analyst, and the supervisor agent that will be used later.
    - Navigate to the folder **agent_config** and inspect the agent contents for each: **business_data_analyst_agent.yaml**, **data_analyst_agent.yaml**, and **supervisor_agent.yaml**

In [0]:
import yaml
from pathlib import Path

agent_files = {
    "da": "data_analyst_agent",
    "ba": "business_data_analyst_agent",
    "sa": "supervisor_agent",
}

agents = {}
for prefix, filename in agent_files.items():
    cfg = yaml.safe_load(Path(f"./agent_config/{filename}.yaml").read_text())
    agents[prefix] = {
        "name": cfg["name"],
        "llm_endpoint": cfg["llm_endpoint"],
        "system_prompt": cfg["system_prompt"],
    }

#### B2.2 Build the Model Service Name

As part of the classroom setup, a Unity Catalog **model service** named `agent-training-model` was created in your catalog and schema. It routes requests through Unity Gateway to `databricks-gpt-5-2`. Agents reference the service, not the model, by its fully qualified name, `catalog.schema.service`, just as in notebook 05. The YAML configs store only the service name in `llm_endpoint`.

**TODO:** Build `model_fqn` from your catalog, your schema, and the `llm_endpoint` value. All three agents use the same service, so you only need to build it once.

In [0]:
## Fully qualified model service name: catalog.schema.service
# model_fqn = f"<FILL_IN>.<FILL_IN>.{agents['da']['llm_endpoint']}"
model_fqn = f"{catalog_name}.{schema_name}.{agents['da']['llm_endpoint']}"
print(f"Model service: {model_fqn}")

Model service: labuser17579966_1791918748.agent_training.agent-training-model



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
model_fqn = f"{catalog_name}.{schema_name}.{agents['da']['llm_endpoint']}"
print(f"Model service: {model_fqn}")
```

</div>
</details>

#### B2.3 Data Analyst Agent

**TODO:** Create an `McpServer` scoped to the `classify_price_tier` UC function, then create the Data Analyst agent from its YAML config.
- `McpServer.from_uc_function()` takes your catalog, your schema, and the function name.
- Pass the MCP server to the agent with `mcp_servers=` (a list) instead of `tools=`. Scoping each agent to its own MCP server means it can only see the tool it needs.
- Use `agents["da"]["system_prompt"]` for the instructions and `model_fqn` for the model.

In [0]:
## MCP server scoped to a single UC function
# da_mcp_server = McpServer.<FILL_IN>(
#     catalog=<FILL_IN>,
#     schema=<FILL_IN>,
#     function_name=<FILL_IN>,
#     workspace_client=ws,
#     timeout=60.0,
# )
da_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="classify_price_tier",
    workspace_client=ws,
    timeout=60.0,
)

## Data Analyst agent, backed by the MCP server above
# data_analyst_agent = Agent(
#     name=agents["da"]["name"],
#     instructions=<FILL_IN>,
#     model=<FILL_IN>,
#     mcp_servers=<FILL_IN>,
# )
data_analyst_agent = Agent(
    name=agents["da"]["name"],
    instructions=agents["da"]["system_prompt"],
    model=model_fqn,
    mcp_servers=[da_mcp_server],
)
print(f"Created agent: {data_analyst_agent.name}")

Created agent: Data Analyst



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
da_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="classify_price_tier",
    workspace_client=ws,
    timeout=60.0,
)

data_analyst_agent = Agent(
    name=agents["da"]["name"],
    instructions=agents["da"]["system_prompt"],
    model=model_fqn,
    mcp_servers=[da_mcp_server],
)
print(f"Created agent: {data_analyst_agent.name}")
```

</div>
</details>

#### B2.4 Business Analyst Agent

**TODO:** Repeat the same pattern for the Business Analyst, this time writing the full calls yourself:
- An `McpServer` scoped to the `neighborhood_summary` UC function, stored in `ba_mcp_server`
- An `Agent` built from the `agents["ba"]` config, stored in `business_analyst_agent`

In [0]:
## MCP server scoped to the neighborhood_summary UC function
# ba_mcp_server = <FILL_IN>
ba_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="neighborhood_summary",
    workspace_client=ws,
    timeout=60.0,
)

## Business Analyst agent, backed by ba_mcp_server
# business_analyst_agent = <FILL_IN>
business_analyst_agent = Agent(
    name=agents["ba"]["name"],
    instructions=agents["ba"]["system_prompt"],
    model=model_fqn,
    mcp_servers=[ba_mcp_server],
)

print(f"Created agent: {business_analyst_agent.name}")

Created agent: Business Analyst



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
ba_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="neighborhood_summary",
    workspace_client=ws,
    timeout=60.0,
)

business_analyst_agent = Agent(
    name=agents["ba"]["name"],
    instructions=agents["ba"]["system_prompt"],
    model=model_fqn,
    mcp_servers=[ba_mcp_server],
)

print(f"Created agent: {business_analyst_agent.name}")
```

</div>
</details>

## C. Create a Supervisor Agent with Handoffs

### C1. Build the Supervisor

The supervisor agent analyzes each user request and delegates to the appropriate worker using the **handoff** mechanism from the [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents). When the supervisor hands off to a worker, that worker takes over the conversation and responds directly to the user.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The <code>handoffs</code> parameter tells the supervisor which agents it can delegate to. The supervisor's instructions guide its routing decisions. The LLM reads each worker's name and instructions to decide which one is best suited for the request.
See: <a href="https://developers.openai.com/api/docs/guides/agents/orchestration" style="color: #1976d2; text-decoration: underline;">Orchestration and handoffs</a>
</p>
</div>
</div>
</div>

**TODO:** Create the supervisor and give it both worker agents as handoffs. The supervisor has no tools of its own: its only job is to pick a worker.

In [0]:
# supervisor = Agent(
#     name=agents["sa"]["name"],
#     instructions=agents["sa"]["system_prompt"],
#     model=model_fqn,
#     <FILL_IN>=[<FILL_IN>, <FILL_IN>],
# )

supervisor = Agent(
    name=agents["sa"]["name"],
    instructions=agents["sa"]["system_prompt"],
    model=model_fqn,
    handoffs=[data_analyst_agent, business_analyst_agent],
)

print(f"Created supervisor with handoffs to: {[a.name for a in supervisor.handoffs]}")

Created supervisor with handoffs to: ['Data Analyst', 'Business Analyst']



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
supervisor = Agent(
    name=agents["sa"]["name"],
    instructions=agents["sa"]["system_prompt"],
    model=model_fqn,
    handoffs=[data_analyst_agent, business_analyst_agent],
)

print(f"Created supervisor with handoffs to: {[a.name for a in supervisor.handoffs]}")
```

</div>
</details>

### C2. How the Supervisor Routes Requests

The supervisor sees each worker as a **handoff tool**. When a user sends a request, the supervisor's LLM reads the request, examines the available handoffs, and decides which worker should handle it.

**Click on any component in the diagram** to learn more about it.

<style>
.ma-node {
  border-radius: 14px; padding: 12px 18px; text-align: center; font-weight: 600; font-size: 14px;
  cursor: pointer; user-select: none; transition: transform 0.12s, box-shadow 0.15s;
  box-shadow: 0 2px 8px rgba(27,49,57,.07);
}
.ma-node:hover { transform: translateY(-2px); }
.ma-node.active { box-shadow: 0 4px 14px rgba(27,49,57,.14); transform: translateY(-2px); }
.ma-arrow { color: #999; font-size: 22px; padding: 0 8px; user-select: none; }
.ma-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.ma-detail-wrap.open { max-height: 1200px; opacity: 1; margin-top: 16px; }
.ma-detail-card { background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-size: 14pt; line-height: 1.7; color: #333; }
.ma-detail-card strong { color: #0b2026; }
.ma-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 13pt; }
.ma-example { background: #F9F7F4; color: #1B3139; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin-top: 12px; font-family: 'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size: 0.82rem; line-height: 1.6; }
.ma-example .q { color: #2563EB; } .ma-example .a { color: #059669; } .ma-example .r { color: #D97706; }
</style>

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:18px 0;flex-wrap:wrap;">
  <div class="ma-node" style="background:transparent;color:#0b2026;border:2px solid #0b2026;" data-ma="input" onclick="maSelect('input')">
    User Query
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div class="ma-node" id="ma-supervisor" style="background:#FF5F46;color:#fff;" data-ma="supervisor" onclick="maSelect('supervisor')">
    Supervisor<br/><span style="font-weight:400;font-size:12px;">Reads instructions + handoffs</span>
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div style="display:flex;flex-direction:column;gap:6px;">
    <div class="ma-node" style="background:#2574B5;color:#fff;" data-ma="data" onclick="maSelect('data')">
      Data Analyst<br/><span style="font-weight:400;font-size:12px;">classify_price_tier via MCP</span>
    </div>
    <div class="ma-node" style="background:#02A36F;color:#fff;" data-ma="business" onclick="maSelect('business')">
      Business Analyst<br/><span style="font-weight:400;font-size:12px;">neighborhood_summary via MCP</span>
    </div>
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div class="ma-node" style="background:transparent;color:#0b2026;border:2px solid #0b2026;" data-ma="output" onclick="maSelect('output')">
    Response
  </div>
</div>

<div class="ma-detail-wrap" id="ma-detail-wrap">
  <div class="ma-detail-card" id="ma-detail-card"></div>
</div>

</div>

<script>
var MA_DATA = {
  input: {
    color: '#1B5162',
    title: 'User Query (Input)',
    html: '<strong>What happens here:</strong> The user sends a natural language question. The system does not pre-classify the query. The raw text is passed directly to the supervisor agent.<br/><br/><strong>Examples:</strong><div class="ma-example"><span class="q">"Is $200/night a good deal for an Airbnb in SF?"</span> &#x2192; routes to Data Analyst<br/><span class="q">"Which neighborhoods have the highest average price?"</span> &#x2192; routes to Business Analyst</div>'
  },
  supervisor: {
    color: '#FF5F46',
    title: 'Supervisor Agent',
    html: '<strong>Role:</strong> Analyzes every incoming request and decides which worker should handle it.<br/><br/><strong>How it decides:</strong> The supervisor has <code>handoffs=[data_analyst_agent, business_analyst_agent]</code>. The LLM sees each handoff as a callable tool named <code>transfer_to_Data_Analyst</code> and <code>transfer_to_Business_Analyst</code>. Based on the user query and the worker descriptions in its instructions, it picks the best match.<br/><br/><strong>Example routing:</strong><div class="ma-example"><span class="q">User: "Is $200/night a good deal in SF?"</span><br/><span class="r">Supervisor thinks: This is about price classification...</span><br/><span class="a">Action: transfer_to_Data_Analyst</span></div><div class="ma-example" style="margin-top:8px;"><span class="q">User: "Which neighborhoods have the highest average price?"</span><br/><span class="r">Supervisor thinks: This is a market/business question about neighborhoods...</span><br/><span class="a">Action: transfer_to_Business_Analyst</span></div>'
  },
  data: {
    color: '#2574B5',
    title: 'Data Analyst Agent',
    html: '<strong>Role:</strong> Classifies Airbnb listing prices into market tiers based on SF pricing distribution.<br/><br/><strong>Tool:</strong> <code>classify_price_tier(price: float)</code> accessed via MCP from Unity Catalog. Returns a JSON object with tier (Budget, Mid-Range, Premium, Luxury), percentile band, and interpretation.<br/><br/><strong>Example interaction:</strong><div class="ma-example"><span class="q">User: "Is $200/night a good deal in SF?"</span><br/><span class="r">Agent calls: classify_price_tier(200.0)</span><br/><span class="a">Agent: "$200/night falls in the Premium tier (50-75th percentile). This is above average for SF listings."</span></div>'
  },
  business: {
    color: '#02A36F',
    title: 'Business Analyst Agent',
    html: '<strong>Role:</strong> Surfaces market insights for stakeholders by analyzing neighborhood-level pricing and listing mix data.<br/><br/><strong>Tool:</strong> <code>neighborhood_summary(neighborhood: str)</code> accessed via MCP from Unity Catalog. Returns average price, total listings, and most common room type per neighborhood.<br/><br/><strong>Example interaction:</strong><div class="ma-example"><span class="q">User: "Which neighborhoods have the highest average price?"</span><br/><span class="r">Agent calls: neighborhood_summary()</span><br/><span class="a">Agent: "The top neighborhoods by average price are: Sea Cliff ($284.50, 89% entire homes), Pacific Heights ($263.15, 72% entire homes), and Nob Hill ($241.80, 65% entire homes)."</span></div>'
  },
  output: {
    color: '#1B5162',
    title: 'Response (Output)',
    html: '<strong>What happens here:</strong> The worker agent that received the handoff produces the final response. The response goes directly back to the user.<br/><br/><strong>Key detail:</strong> The supervisor does NOT post-process or modify the worker output. Once a handoff occurs, the worker owns the conversation. This is why <code>result.last_agent.name</code> shows the worker, not the supervisor.<br/><br/><strong>In the trace:</strong> The MLflow trace captures the full flow: supervisor LLM call &#x2192; handoff decision &#x2192; worker LLM call &#x2192; tool calls (if any) &#x2192; final output.'
  }
};
var maCurrent = null;
function maSelect(id) {
  var wrap = document.getElementById('ma-detail-wrap');
  var card = document.getElementById('ma-detail-card');
  document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.toggle('active', n.dataset.ma === id); });
  if (maCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.remove('active'); }); maCurrent = null; return; }
  maCurrent = id;
  var d = MA_DATA[id];
  card.innerHTML = '<div style="height:5px;border-radius:5px;margin:-4px 0 12px;background:' + d.color + ';"></div><div style="font-size:16pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div style="font-size:15pt;color:#0b2026;line-height:1.7;">' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## D. Test Multi-Agent Routing

### D1. Data Question

This question is about price classification. The supervisor should route it to the Data Analyst agent, which will call `classify_price_tier` via MCP. You can verify this in MLflow interface as a part of running the next cell. 

We are connecting to the Databricks managed MCP server via`McpServer`, which is an async context manager, hence we need to be mindful of using `async` and `await`. Similarly, `Runner.run()` is also async.

**TODO:** Fill in:
- both MCP servers in the `async with` statement, since they must stay open while the agents run
- the `Runner` method that runs the supervisor (the same one you used in notebook 05)
- the `result` attributes for the agent that produced the final answer and for the final response text

In [0]:
data_query = "Is $200/night a good deal for an Airbnb in SF?"

# async def test_data_query():
#     async with <FILL_IN>, <FILL_IN>:
#         result = await <FILL_IN>(supervisor, data_query)
#         return result
    
async def test_data_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, data_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_data_query())
print(f"User:       {data_query}")
# print(f"Handled by: {result.<FILL_IN>.name}")
# print(f"Response:   {result.<FILL_IN>}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

User:       Is $200/night a good deal for an Airbnb in SF?
Handled by: Data Analyst
Response:   $200/night in San Francisco is **Premium** tier, sitting around the **50–75th percentile** of SF Airbnb nightly prices—i.e., **above average**, not a “steal.”

Whether it’s a *good deal* depends on what you’re getting:
- **Good deal at $200**: entire place, strong location (Downtown/SoMa/Marina/Mission), newer/renovated, parking included.
- **Not a great deal at $200**: small private room, weaker location, extra fees that push the effective nightly cost much higher.

If you share the neighborhood, dates (seasonality matters), and whether it’s an entire place vs. private room, I can sanity-check value more precisely.


Trace(trace_id=tr-023cc8e08bd97b289d6237acbe4d1f0b)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
data_query = "Is $200/night a good deal for an Airbnb in SF?"

async def test_data_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, data_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_data_query())
print(f"User:       {data_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")
```

</div>
</details>

### D2. Business Question

**TODO:** Write your own question that the supervisor should hand off to the Business Analyst, which answers it by calling `neighborhood_summary` via MCP. Look at the Business Analyst's routing rules in `agent_config/supervisor_agent.yaml` for ideas, then check `Handled by` in the output to confirm the handoff.

In [0]:
## Write a question the supervisor should route to the Business Analyst
# business_query = "<FILL_IN>"
business_query = "What is the average price of the Mission neighborhood?"

async def test_business_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, business_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_business_query())
print(f"User:       {business_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

User:       What is the average price of the Mission neighborhood?
Handled by: Business Analyst
Response:   The **average nightly price in the Mission neighborhood is $188.63** (across **8 listings**). The **most common room type** is **Entire home/apt**.


Trace(trace_id=tr-96ce1e5d2c76020f812b879473c913a1)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
## Any neighborhood-level question works, for example:
business_query = "What is the average price of the Mission neighborhood?"

async def test_business_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, business_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_business_query())
print(f"User:       {business_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")
```

</div>
</details>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
Why are there traces in the outputs above?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">
<p style="margin-top: 10px;">Running the cell above will produce a trace showing a tool call took place. This is because <code>@mlflow.trace(span_type=SpanType.TOOL)</code> is used during <code>call_tool</code> (see the <a href="https://github.com/databricks/databricks-ai-bridge/blob/main/integrations/openai/src/databricks_openai/agents/mcp_server.py" style="color: #1976d2; text-decoration: underline;">source code</a>).</p>
</div>
</details>

### D3. Mixed Routing Test

Run multiple queries and show how the supervisor routes each one to the appropriate worker agent. Notice that in the output there are multiple traces. In the next section, we will see how to concatenate them into 1 trace and perform proper logging as a series of events so we have a more comprehensive picture of the flow from input to output via the supervisor agent.

**TODO:** For each query, fill in the agent you expect the supervisor to hand off to, using `agents["da"]["name"]` or `agents["ba"]["name"]`. The last query combines a price with a neighborhood, so think about which tool can actually answer it: `classify_price_tier` only knows citywide price tiers, while `neighborhood_summary` knows each neighborhood's average price.

Routing is an LLM decision, so a `MISMATCH` can appear on any run. You will tighten the routing rules in the next step.

In [0]:
import pandas as pd

## Expected agent for each query: agents["da"]["name"] or agents["ba"]["name"]
# test_queries = [
#     ("Is $150/night a good deal in SF?", <FILL_IN>),
#     ("What is the average price in Mission?", <FILL_IN>),
#     ("Would $350/night be considered luxury?", <FILL_IN>),
#     ("Is the average price in Mission higher or lower than in Noe Valley?", <FILL_IN>),
#     ("Is $250/night expensive for the Mission?", <FILL_IN>),
# ]
test_queries = [
    ("Is $150/night a good deal in SF?", agents["da"]["name"]),
    ("What is the average price in Mission?", agents["ba"]["name"]),
    ("Would $350/night be considered luxury?", agents["da"]["name"]),
    ("Is the average price in Mission higher or lower than in Noe Valley?", agents["ba"]["name"]),
    ("Is $250/night expensive for the Mission?", agents["ba"]["name"]),
]

async def test_mixed_routing():
    async with da_mcp_server, ba_mcp_server:
        results = []
        for query, expected in test_queries:
            result = await Runner.run(supervisor, query)
            actual = result.last_agent.name
            status = "PASS" if actual == expected else "MISMATCH"
            results.append({
                "Query": query,
                "Expected Agent": expected,
                "Actual Agent": actual,
                "Status": status,
            })
        return results

results = asyncio.get_event_loop().run_until_complete(test_mixed_routing())
display(pd.DataFrame(results))

,Query,Expected Agent,Actual Agent,Status
0,Is $150/night a good deal in SF?,Data Analyst,Data Analyst,PASS
1,What is the average price in Mission?,Business Analyst,Business Analyst,PASS
2,Would $350/night be considered luxury?,Data Analyst,Data Analyst,PASS
3,Is the average price in Mission higher or lowe...,Business Analyst,Business Analyst,PASS
4,Is $250/night expensive for the Mission?,Business Analyst,Data Analyst,MISMATCH


[Trace(trace_id=tr-e3de638a47181c0c069ac5cd8ab5f05a), Trace(trace_id=tr-625958ce2fdfafef8a298d1c74847bbd), Trace(trace_id=tr-4c48b13a5c19d3b4c739e84b5e15a171), Trace(trace_id=tr-3e2f08d4e2f55f84fc40448bd96d6c5a), Trace(trace_id=tr-a263c56620f99f4bdbfe1b9804306028), Trace(trace_id=tr-04ddb0c7c1957978c724ed5678ad9131)]


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import pandas as pd

test_queries = [
    ("Is $150/night a good deal in SF?", agents["da"]["name"]),
    ("What is the average price in Mission?", agents["ba"]["name"]),
    ("Would $350/night be considered luxury?", agents["da"]["name"]),
    ("Is the average price in Mission higher or lower than in Noe Valley?", agents["ba"]["name"]),
    ## Needs the Mission's average price, which only neighborhood_summary provides
    ("Is $250/night expensive for the Mission?", agents["ba"]["name"]),
]

async def test_mixed_routing():
    async with da_mcp_server, ba_mcp_server:
        results = []
        for query, expected in test_queries:
            result = await Runner.run(supervisor, query)
            actual = result.last_agent.name
            status = "PASS" if actual == expected else "MISMATCH"
            results.append({
                "Query": query,
                "Expected Agent": expected,
                "Actual Agent": actual,
                "Status": status,
            })
        return results

results = asyncio.get_event_loop().run_until_complete(test_mixed_routing())
display(pd.DataFrame(results))
```

</div>
</details>

### D4. Refine the Supervisor's Routing Instructions

The supervisor's routing rules come from its instructions in `agent_config/supervisor_agent.yaml`, not from your code. This file is in your copy of the course folder, so you can edit it. The last query in D3 combines a price with a neighborhood, and the current rules don't say which worker owns that kind of question. Fix that in the instructions.

**TODO:**
1. Open `agent_config/supervisor_agent.yaml` from the workspace file browser.
1. Under the `Route to Business Analyst for:` list in `system_prompt`, add a rule for questions that compare a specific price to a specific neighborhood. Keep the same indentation as the existing lines, because `system_prompt` is a YAML block.
1. Save the file, then run the next cell. It reloads the YAML, rebuilds the supervisor with your new instructions, and re-runs the D3 tests.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```yaml
Route to Business Analyst for:
  - Neighborhood summaries and comparisons
  - Average prices by area
  - Listing counts and room type mix
  - Market trends and insights
  - Comparing a specific price to a specific neighborhood (for example,
    "Is $250/night expensive for the Mission?"), because answering requires
    that neighborhood's average price from neighborhood_summary
```

</div>
</details>

In [0]:
## Reload the edited supervisor config, rebuild the supervisor, and re-run the D3 routing tests
cfg = yaml.safe_load(Path("./agent_config/supervisor_agent.yaml").read_text())
agents["sa"]["system_prompt"] = cfg["system_prompt"]

supervisor = Agent(
    name=agents["sa"]["name"],
    instructions=agents["sa"]["system_prompt"],
    model=model_fqn,
    handoffs=[data_analyst_agent, business_analyst_agent],
)

results = asyncio.get_event_loop().run_until_complete(test_mixed_routing())
display(pd.DataFrame(results))

,Query,Expected Agent,Actual Agent,Status
0,Is $150/night a good deal in SF?,Data Analyst,Business Analyst,MISMATCH
1,What is the average price in Mission?,Business Analyst,Business Analyst,PASS
2,Would $350/night be considered luxury?,Data Analyst,Data Analyst,PASS
3,Is the average price in Mission higher or lowe...,Business Analyst,Business Analyst,PASS
4,Is $250/night expensive for the Mission?,Business Analyst,Business Analyst,PASS


[Trace(trace_id=tr-49b790017e4b32b16b5a9fc968514de7), Trace(trace_id=tr-2fc89b11dff785bc6d7229d52dc2bdd5), Trace(trace_id=tr-f29f85717fd7047f3df5adb46a3148db), Trace(trace_id=tr-dbccb92099e2164cf840cec8b63a503d), Trace(trace_id=tr-7cbab97afe2a7320febde8728b2890b0), Trace(trace_id=tr-01a0af448cd658bcd91878d36ddb91be)]

## E. Enable MLflow Tracing
Now that we have built and validated our multi-agent system, we enable autologging with MLflow to capture execution traces. This section will have fractured traces showing each handoff and tool call as a separate trace. We will see later how we can bring these together in 1 trace. For now, let's focus on the individual outputs to see how a multi-agent system appears as multiple traces. 

### E1. Enable Autologging

We enable MLflow tracing so the next agent runs produce full execution traces.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
In the MLflow Experiment UI, multi-agent traces show nested spans: the supervisor's LLM call appears at the top level, followed by the handoff event, and then the worker's LLM and tool calls as child spans. This makes it easy to debug routing decisions and tool selection.
</p>
</div>
</div>
</div>

**TODO:** Turn on MLflow autologging for the OpenAI integration, as you did in notebook 05 (section D1).

In [0]:
import mlflow

# mlflow.<FILL_IN>.<FILL_IN>()
mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")
print(f"Traces will appear in experiment: {mlflow_experiment_path}")

MLflow OpenAI autologging enabled.
Traces will appear in experiment: /Workspace/Users/labuser17579966_1791918748@vocareum.com/oai-ma-sdk-training



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow

mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")
print(f"Traces will appear in experiment: {mlflow_experiment_path}")
```

</div>
</details>

### E2. Run with Tracing

Run the supervisor again with tracing enabled. The trace will capture the full handoff flow from supervisor to worker. The `transfer_to_Data_Analyst` and `transfer_to_Business_Analyst` tools do not exist in your code. The SDK generates them automatically from `handoffs=[data_analyst_agent, business_analyst_agent]` using the pattern `transfer_to_{agent.name}`.

In [0]:
async def run_with_tracing(query: str):
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, query)
        return result

result = asyncio.get_event_loop().run_until_complete(run_with_tracing(data_query))
print(f"User:       {data_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

User:       Is $200/night a good deal for an Airbnb in SF?
Handled by: Data Analyst
Response:   $200/night in San Francisco is **Premium tier** pricing.

- **Tier:** Premium  
- **Market position:** **50–75th percentile** (above average)  
- **Verdict:** It’s **not a “good deal” on price alone**—it’s **above-average** for SF. It can still be good value if it has strong reviews, a great location, parking, or more space/amenities than typical listings.


Trace(trace_id=tr-dc2e24fccee7b8632d30380a988a8983)

## F. Distributed Tracing

Notice the output from the previous section may produce multiple separate trace outputs. MLflow supports distributed tracing to stitch spans from different calls into a single unified trace.

The key idea: wrap the entire multi-agent flow in a parent `@mlflow.trace` span. All child spans (from autologging, tool calls, and handoffs) automatically nest under it, producing one clean trace tree instead of scattered fragments.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
For cross-service distributed tracing (e.g., an agent calling another agent over HTTP), MLflow provides <code>get_tracing_context_headers_for_http_request()</code> and <code>set_tracing_context_from_http_request_headers()</code> to propagate trace context via W3C TraceContext headers. Both services must log to the same MLflow tracking server and experiment. On Databricks, the trace destination must be set to Unity Catalog.
See: <a href="https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/distributed-tracing/" style="color: #1976d2; text-decoration: underline;">MLflow Distributed Tracing docs</a>
</p>
</div>
</div>
</div>

### F1. Build Traced Agent Wrappers

To make handoffs visible in the trace, we wrap each agent's execution in its own `@mlflow.trace` span. The supervisor routes the question, then we call the selected worker inside a labeled span. This produces a trace tree like:

```
supervisor_orchestration
  +-- route_decision (supervisor LLM call)
  +-- data_analyst_execution OR business_analyst_execution
      +-- AsyncDatabricksCompletions (worker LLM + tool calls)
```

1. Create a trace wrapper for the agents defined above. Here we are setting the `span_type` to `AGENT` and defining the names of the agents as they will appear in the trace. Again, we are connecting to the Databricks managed MCP server via`McpServer`, which is an async context manager, hence we need to be mindful of using `async` and `await` as mentioned in section **D**.

**TODO:** Fill in:
- the span type for each wrapper (the same span type you used for the agent wrapper in notebook 05, section D4)
- the MCP server each worker needs open while it runs
- the agent each wrapper runs

In [0]:
## Traced wrapper for the Data Analyst
# @mlflow.trace(span_type=<FILL_IN>, name="data_analyst_execution")
# async def run_data_analyst(question: str) -> str:
#     async with <FILL_IN>:
#         result = await Runner.run(<FILL_IN>, question)
#         return result.final_output
@mlflow.trace(span_type="AGENT", name="data_analyst_execution")
async def run_data_analyst(question: str) -> str:
    async with da_mcp_server:
        result = await Runner.run(data_analyst_agent, question)
        return result.final_output

## Traced wrapper for the Business Analyst
# @mlflow.trace(span_type=<FILL_IN>, name="business_analyst_execution")
# async def run_business_analyst(question: str) -> str:
#     async with <FILL_IN>:
#         result = await Runner.run(<FILL_IN>, question)
#         return result.final_output
@mlflow.trace(span_type="AGENT", name="business_analyst_execution")
async def run_business_analyst(question: str) -> str:
    async with ba_mcp_server:
        result = await Runner.run(business_analyst_agent, question)
        return result.final_output




<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
@mlflow.trace(span_type="AGENT", name="data_analyst_execution")
async def run_data_analyst(question: str) -> str:
    async with da_mcp_server:
        result = await Runner.run(data_analyst_agent, question)
        return result.final_output

@mlflow.trace(span_type="AGENT", name="business_analyst_execution")
async def run_business_analyst(question: str) -> str:
    async with ba_mcp_server:
        result = await Runner.run(business_analyst_agent, question)
        return result.final_output
```

</div>
</details>

2. Define the router and the parent trace. The router is a plain agent with **no handoffs**. It reuses the supervisor's instructions, so instead of handing off it replies with the name of the worker that should handle the request, and we read that reply as the route. The parent `@mlflow.trace` span, `supervisor_orchestration`, is provided.

**TODO:** Fill in:
- the `mlflow` function that opens a child span named `route_decision` inside the parent trace
- the span methods that record the span's inputs and outputs
- the traced worker wrapper from step 1 that matches each route

In [0]:
## Router agent (no handoffs, just classifies the request)
router_agent = Agent(
    name="Router",
    instructions=agents["sa"]["system_prompt"],
    model=model_fqn,
)


@mlflow.trace(span_type="AGENT", name="supervisor_orchestration")
async def run_supervisor_traced(question: str) -> str:
    """Route to the correct agent and execute, with each step visible in the trace."""

    # Step 1: Record the routing decision as its own child span

    # with mlflow.<FILL_IN>(name="route_decision") as span:
    #     span.<FILL_IN>({"question": question})
    #     route_result = await Runner.run(router_agent, question)
    #     route = route_result.final_output.strip().lower()
    #     span.<FILL_IN>({"route": route})

    with mlflow.start_span(name="route_decision") as span:
        span.set_inputs({"question": question})
        route_result = await Runner.run(router_agent, question)
        route = route_result.final_output.strip().lower()
        span.set_outputs({"route": route})

    # Step 2: Run the traced worker wrapper that matches the route
    if "business" in route:
        # return await <FILL_IN>(question)
        return await run_business_analyst(question)
    else:
        # return await <FILL_IN>(question)
        return await run_data_analyst(question)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
router_agent = Agent(
    name="Router",
    instructions=agents["sa"]["system_prompt"],
    model=model_fqn,
)


@mlflow.trace(span_type="AGENT", name="supervisor_orchestration")
async def run_supervisor_traced(question: str) -> str:
    """Route to the correct agent and execute, with each step visible in the trace."""

    # Step 1: Record the routing decision as its own child span
    with mlflow.start_span(name="route_decision") as span:
        span.set_inputs({"question": question})
        route_result = await Runner.run(router_agent, question)
        route = route_result.final_output.strip().lower()
        span.set_outputs({"route": route})

    # Step 2: Run the traced worker wrapper that matches the route
    if "business" in route:
        return await run_business_analyst(question)
    else:
        return await run_data_analyst(question)
```

</div>
</details>

### F2. Run and Inspect Unified Traces

1. Run the next cell, then open the trace in the MLflow experiment. The parent span, `supervisor_orchestration`, is the caller. The worker span, such as `data_analyst_execution`, is the callee that calls the MCP server. The routing decision appears as its own `route_decision` span between them.

In [0]:
# Run a data question through the traced wrapper
response = await run_supervisor_traced(data_query)
print(f"Question: {data_query}")
print(f"Response: {response}")

Question: Is $200/night a good deal for an Airbnb in SF?
Response: $200/night in San Francisco is **Premium tier**—around the **50th–75th percentile** of SF Airbnb prices.

- **Deal?** It’s **not a “cheap” deal**, but it can be a **good value** if it’s a strong listing (great location, entire place, newer/renovated, strong reviews) since it’s **above the SF average** rather than below it.


Trace(trace_id=tr-94479c6947c5cdf516f34e5a328f70c6)

2. As a final example, we pass a similar query to the business analyst agent. 

In [0]:
# Run a business question through the traced wrapper
response = await run_supervisor_traced(business_query)
print(f"Question: {business_query}")
print(f"Response: {response}")

Question: What is the average price of the Mission neighborhood?
Response: The **average nightly price in the Mission** is **$188.63**.

Quick context: there are **8 listings** in the dataset for the Mission, and the dominant room type is **Entire home/apt**.


Trace(trace_id=tr-dbbc858f31510573ca2abc0349aa8278)

## G. Conclusion
In this lab, you:

1. Configured the OpenAI Agents SDK to route requests through Databricks model services
2. Discovered UC functions through a Managed MCP server and connected each worker agent to a single function with `McpServer`
3. Built a supervisor agent that delegates user requests to the Data Analyst and Business Analyst with `handoffs`
4. Tested routing across data, business, and mixed queries, then refined the routing by editing the supervisor's YAML instructions
5. Captured multi-agent runs with MLflow autologging
6. Combined the supervisor and worker spans into a single trace with `@mlflow.trace` and `mlflow.start_span`

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>